In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [33]:
FEATURES = [
    "ALLSKY_SFC_SW_DIFF", "ALLSKY_SFC_SW_DNI", "TOA_SW_DWN",
    "RH2M", "QV2M", "PS", "WS2M", "CLOUD_AMT",
    "ALLSKY_SFC_LW_DWN", "T2M", 
]

In [30]:
from final.final_fetch import evaluation_fetch

In [31]:
df:pd.DataFrame = evaluation_fetch(19.0760, 72.8777,"777628119ce049d484833355dbeca175")

2025-08-07 00:00:00
2025-08-10 00:00:00
https://api.weatherbit.io/v2.0/history/hourly?lat=19.076&lon=72.8777&key=777628119ce049d484833355dbeca175&start_date=2025-08-07%3A00&end_date=2025-08-10%3A00
Index(['app_temp', 'azimuth', 'clouds', 'datetime', 'dewpt', 'dhi', 'dni',
       'elev_angle', 'ghi', 'h_angle', 'pod', 'precip', 'pres',
       'revision_status', 'revision_version', 'rh', 'slp', 'snow', 'solar_rad',
       'temp', 'timestamp_local', 'ts', 'uv', 'vis', 'weather', 'wind_dir',
       'wind_gust_spd', 'wind_spd', 'hour', 'month', 'hour_sin', 'hour_cos',
       'month_sin', 'month_cos', 'RH2M', 'PS', 'T2M', 'WS2M', 'CLOUD_AMT',
       'ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'QV2M',
       'TOA_SW_DWN', 'ALLSKY_SFC_LW_DWN'],
      dtype='object')


In [20]:
df.head()

,app_temp,azimuth,clouds,datetime,dewpt,dhi,dni,elev_angle,ghi,h_angle,...,PS,T2M,WS2M,CLOUD_AMT,ALLSKY_SFC_SW_DWN,ALLSKY_SFC_SW_DNI,ALLSKY_SFC_SW_DIFF,QV2M,TOA_SW_DWN,ALLSKY_SFC_LW_DWN
timestamp_utc,,,,,,,,,,,,,,,,,,,,,
2025-08-07 00:00:00+00:00,23.4,68.4,69,2025-08-07:00,21.9,0,0,-11.1,0,None,...,932,22.6,0.4,69,0,0,0,0.017752,0,389.908801
2025-08-07 01:00:00+00:00,23.0,73.5,73,2025-08-07:01,21.9,21,149,2.3,18,None,...,932,22.2,0.4,73,18,149,21,0.017685,32,388.056752
2025-08-07 02:00:00+00:00,23.5,77.8,59,2025-08-07:02,22.3,66,606,16.1,227,None,...,933,22.6,0.8,59,227,606,66,0.018106,174,390.163247
2025-08-07 03:00:00+00:00,24.5,81.4,58,2025-08-07:03,22.1,90,770,30.1,470,None,...,934,23.7,0.8,58,470,770,90,0.017949,368,395.075687
2025-08-07 04:00:00+00:00,25.2,84.9,44,2025-08-07:04,22.1,105,856,44.2,696,None,...,934,24.4,0.8,44,696,856,105,0.017897,614,398.254015


TypeError: 'generator' object is not subscriptable

In [34]:
# ...existing code...
for i in FEATURES:
    df_2 = pd.read_csv('dataset/mumbai_' + i + '_hourly.csv')
    mean1 = df_2[i].iloc[120:200].mean()
    mean2 = df[i].mean()
    print(f"{i} mean1 (Mumbai 120:200): {mean1:.4f}, mean2 (df): {mean2:.4f}")
# ...existing code...

ALLSKY_SFC_SW_DIFF mean1 (Mumbai 120:200): 90.0421, mean2 (df): 51.3750
ALLSKY_SFC_SW_DNI mean1 (Mumbai 120:200): 10.6536, mean2 (df): 403.8194
TOA_SW_DWN mean1 (Mumbai 120:200): 408.8175, mean2 (df): 209.3194
RH2M mean1 (Mumbai 120:200): 91.1681, mean2 (df): 83.2500
QV2M mean1 (Mumbai 120:200): 19.8909, mean2 (df): 0.0195
PS mean1 (Mumbai 120:200): 99.6469, mean2 (df): 1006.7778
WS2M mean1 (Mumbai 120:200): 2.9985, mean2 (df): 2.9228
CLOUD_AMT mean1 (Mumbai 120:200): 99.4004, mean2 (df): 39.7083
ALLSKY_SFC_LW_DWN mean1 (Mumbai 120:200): 441.5644, mean2 (df): 416.6152
T2M mean1 (Mumbai 120:200): 26.4824, mean2 (df): 27.8764


In [9]:
import os
import sys
from final.final_fetch import fetch_weatherbit_data
import torch
import numpy as np
import pandas as pd
from models_code.model_5 import SpikeAwareHybrid
from sklearn.preprocessing import StandardScaler
import joblib
from final.fetch import fetch_and_return
from datetime import datetime, timedelta

MODEL_PATH = "models/spike_aware_q95_seq24.pth"
SCALER_PATH ="scalers/feature_scaler_seq24.pkl"
SEQ_LEN = 24
FEATURES = [
    "ALLSKY_SFC_SW_DIFF", "ALLSKY_SFC_SW_DNI", "TOA_SW_DWN",
    "RH2M", "QV2M", "PS", "WS2M", "CLOUD_AMT",
    "ALLSKY_SFC_LW_DWN", "T2M", "hour_sin", "hour_cos", "month_sin", "month_cos"
]
TARGET = "ALLSKY_SFC_SW_DWN"



def prepare_input(df, scaler):
    df = df[FEATURES]
    df_scaled = scaler.transform(df)
    tensor = torch.tensor(df_scaled[-SEQ_LEN:], dtype=torch.float32,device='cpu').unsqueeze(0)
    return tensor
def predict(model, input_tensor):
    with torch.no_grad():
        output = model(input_tensor).item()
        return output

In [10]:
model = SpikeAwareHybrid(len(FEATURES))

In [11]:
model.load_state_dict(torch.load(MODEL_PATH))

C:\Users\grins\AppData\Local\Temp\ipykernel_1468\610011160.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH))


<All keys matched successfully>

In [12]:
scaler = joblib.load(SCALER_PATH)

In [13]:
df = fetch_weatherbit_data(18.5246,73.8786,"777628119ce049d484833355dbeca175")

https://api.weatherbit.io/v2.0/history/hourly?lat=18.5246&lon=73.8786&key=777628119ce049d484833355dbeca175&start_date=2025-08-09%3A07&end_date=2025-08-10%3A07
Index(['app_temp', 'azimuth', 'clouds', 'datetime', 'dewpt', 'dhi', 'dni',
       'elev_angle', 'ghi', 'h_angle', 'pod', 'precip', 'pres',
       'revision_status', 'revision_version', 'rh', 'slp', 'snow', 'solar_rad',
       'temp', 'timestamp_local', 'ts', 'uv', 'vis', 'weather', 'wind_dir',
       'wind_gust_spd', 'wind_spd', 'hour', 'month', 'hour_sin', 'hour_cos',
       'month_sin', 'month_cos', 'RH2M', 'PS', 'T2M', 'WS2M', 'CLOUD_AMT',
       'ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'QV2M',
       'TOA_SW_DWN', 'ALLSKY_SFC_LW_DWN'],
      dtype='object')


In [15]:
df = df[FEATURES]

In [18]:
input_tensor = prepare_input(df,scaler)

a:\weather_net\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [20]:
input_tensor.shape

torch.Size([1, 24, 14])

In [26]:
y_pred_logs = predict(model,input_tensor)

In [28]:
np.expm1(y_pred_logs)

np.float64(70.51334742116795)